# TAB 64A: exp64a_diffusion_attention (exp63 base + diffusion_attention) - Colab
# H16-A: Physically-Grounded Green's Diffusion Distance-Decay Attention Bias
# Enforces physical eddy current diffusion decay -gamma_h * ||p_i - p_j||^2 in Self- & Cross-Attention, eliminating far-field noise overfitting.

In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [13]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

TDMS: 81
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/corrosion/corrosion_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v1/rivet_v1_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v2/rivet_v2_gt_mask.npy


In [14]:
!test -d GraduationProject && (cd GraduationProject && git pull --ff-only) || git clone https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git switch Define_PECT_JEPA && git pull --ff-only && git log --oneline -3

remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 21 (delta 18), reused 16 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (21/21), 5.00 KiB | 512.00 KiB/s, done.
From https://github.com/TranAnh35/GraduationProject
   5664992..3a97ac8  Define_PECT_JEPA -> origin/Define_PECT_JEPA
Updating 5664992..3a97ac8
Fast-forward
 .../cross_condition/downstream/train_downstream.py |   4 +-
 src/PECT_JEPA/cross_condition/evaluate.py          | 204 +++++++++++++++++++--
 .../cross_condition/evaluation/visualizations.py   |  59 ++++++
 3 files changed, 255 insertions(+), 12 deletions(-)
Already on 'Define_PECT_JEPA'
Your branch is up to date with 'origin/Define_PECT_JEPA'.
Already up to date.
3a97ac8 (HEAD -> Define_PECT_JEPA, origin/Define_PECT_JEPA) feat(evaluate): generate UDA probability heatmaps and 3-panel comparisons into 4. Domain Adaptation folder
7195c89 fix(evaluate): print UDA compari

In [15]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"


torch 2.11.0+cu130 | cuda: True | sklearn: 1.7.2


In [16]:
!mkdir -p "$DRIVE_RUNS"
!cd GraduationProject && mkdir -p experiments/cross_condition && if [ ! -L experiments/cross_condition/runs ]; then mv experiments/cross_condition/runs/* "$DRIVE_RUNS/" 2>/dev/null; rm -rf experiments/cross_condition/runs && ln -s "$DRIVE_RUNS" experiments/cross_condition/runs && echo LINKED; else echo ALREADY_LINKED; fi
!cd GraduationProject && ls -la experiments/cross_condition/ | grep runs

ALREADY_LINKED
lrwxrwxrwx 1 root root   41 Oct 19 08:29 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [17]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# ===== CONFIG EXP64A (H16-A: Diffusion Attention) =====
EXP_NAME = 'exp64a_diffusion_attention'
BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --inv_start_epoch 11 --inv_ramp_epochs 20 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --inv_cosine true --zc_cond_weight 0.1 --var_gamma 0.5 --offset_weight 0.0 --pool_var_weight 0.0 --device cuda'
EXTRA = '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --diffusion_attention true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05'
os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print(EXP_NAME)
print(EXTRA)

exp64a_diffusion_attention
--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --diffusion_attention true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05


In [18]:
# Symlink runs sang Drive: checkpoint va metrics ghi truc tiep len Google Drive
!mkdir -p "$DRIVE_RUNS"
!if [ -e GraduationProject/experiments/cross_condition/runs ] && [ ! -L GraduationProject/experiments/cross_condition/runs ]; then mv GraduationProject/experiments/cross_condition/runs GraduationProject/experiments/cross_condition/runs_local_bak; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


lrwxrwxrwx 1 root root   41 Oct 19 13:41 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [ ]:
# Cell Train (Fresh Run): chay de huan luyen exp63 (H15)
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

In [ ]:
# ==============================================================================
# Resume training tiep tuc tu DRIVE_RUNS (CHỈ chạy khi session trước đã dừng hẳn)
# ==============================================================================
import os, glob

# 1. Tim run folder trong DRIVE_RUNS (uu tien run moi nhat hoac chi dinh run cu the)
SPECIFIC_RUN = ""  # De trong de tu dong lay run moi nhat, hoac dien vi du: "exp60_contpos_asyminv_fullctx_20261017_223000"

if SPECIFIC_RUN:
    run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN)
else:
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

print(f"[*] Thu muc run tren Drive: {run_dir}")

# 2. Xac dinh checkpoint (uu tien latest de tiep tuc epoch & optimizer/scaler state, fallback sang best)
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")
ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")

if os.path.exists(ckpt_latest):
    resume_ckpt = ckpt_latest
elif os.path.exists(ckpt_best):
    resume_ckpt = ckpt_best
else:
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Checkpoint resume: {resume_ckpt} ({os.path.getsize(resume_ckpt) / 1024**2:.1f} MB)")

# 3. Kiem tra tien do da train tu metrics_epoch.csv
csv_path = os.path.join(run_dir, "evaluation_results", "metrics_epoch.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(run_dir, "metrics_epoch.csv")
if os.path.exists(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
        if len(lines) > 1:
            last_line = lines[-1].split(",")
            print(f"[*] Da hoan thanh epoch: {last_line[0]} | Val AUC gan nhat: {last_line[14] if len(last_line) > 14 else 'N/A'}")

# 4. Dam bao symlink GraduationProject/experiments/cross_condition/runs -> DRIVE_RUNS
os.system(f'mkdir -p GraduationProject/experiments/cross_condition && ln -sfn "{DRIVE_RUNS}" GraduationProject/experiments/cross_condition/runs')

# 5. Export checkpoint path sang environ va thuc hien resume training
os.environ["RESUME_CKPT"] = resume_ckpt
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name "$EXP_NAME" $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1 --resume_checkpoint "$RESUME_CKPT"

In [19]:
# ==============================================================================
# Full Downstream Evaluation + Readouts Diagnostic sau khi train xong
# ==============================================================================
import os, glob

run_dir = os.environ.get('LAST_RUN_DIR', None)
if not run_dir or not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")))
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")

eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest
if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Dang evaluate checkpoint: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda

[*] Dang evaluate checkpoint: /content/drive/MyDrive/PECT_Dataset/runs/exp64a_diffusion_attention_20261019_082930/checkpoints/best_cross_condition_jepa.pt

PECT-JEPA Unified Downstream Benchmark Evaluation
Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp64a_diffusion_attention_20261019_082930/checkpoints/best_cross_condition_jepa.pt
Output Directory: /content/drive/MyDrive/PECT_Dataset/runs/exp64a_diffusion_attention_20261019_082930/evaluation_results
Compute Device:   cuda
Target Splits:    ['train', 'val', 'test']

Discovered 81 total inspection files:
  - Train Split (Base Domain):     12 files
  - Val Split (Base Domain):       4 files
  - Test Split (Held-out OOD):     65 files

STEP 1: Downstream Probes & UDA Models Setup
  [INFO] Loading pre-trained downstream models from: /content/drive/MyDrive/PECT_Dataset/runs/exp64a_diffusion_attention_20261019_082930/downstream_models
  [INFO] Loaded pre-trained CORAL UDA model!
  [INFO] Loaded pre-trained DANN UDA model!
  [I